# 04 — Fine-Tuning

**Goal:** Fine-tune Mistral-7B-Instruct on our political tweets
so it speaks authentically as Democrat and Republican senators.


## 0. Mount Google Drive

**Why Google Drive?**

Colab sessions disconnect randomly. Without Drive:

Upload your `train_ready.jsonl` to Drive before running.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Your project folder in Drive
# Change this path if your folder is named differently
DRIVE_PATH = '/content/drive/MyDrive/political_council'
os.makedirs(DRIVE_PATH, exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/adapter', exist_ok=True)

# Verify your training data is there
data_path = f'{DRIVE_PATH}/train_ready.jsonl'
if os.path.exists(data_path):
    lines = open(data_path).readlines()
    print(f'Training data found: {len(lines):,} examples')
else:
    print('ERROR: train_ready.jsonl not found in Drive')
    print(f'Upload it to: {DRIVE_PATH}/')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Training data found: 10,000 examples


## 1. Install Libraries


In [ ]:
# !pip install transformers peft trl bitsandbytes accelerate datasets -q


In [ ]:
# !pip install -q trl

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer
from datasets import load_dataset

# Verify GPU
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')


GPU: Tesla T4
VRAM: 14.6 GB


## 2. Load Training Dataset

The dataset we prepared in 02 — 10k examples in Mistral format.

We split 90% train / 10% eval.



In [ ]:
# Load dataset from Drive
dataset = load_dataset(
    'json',
    data_files=data_path,
    split='train'
)

print(f'Total examples: {len(dataset):,}')
print(f'Columns: {dataset.column_names}')
print()
print('Sample:')
print(dataset[0]['text'])


Total examples: 10,000
Columns: ['text', 'ideology', 'topic', 'raw_tweet']

Sample:
<s>[INST] You are a Democratic senator speaking about social equality and civil rights. Express your position. [/INST]
Our AAPI communities make our state and our country stronger. This AAPIHeritageMonth we celebrate the contributions of Asian Americans and Pacific Islanders to our nation and we must also continue to stand up and speak out against anti-Asian sentiment and discrimination</s>


In [ ]:
# 90% train, 10% eval
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split['train']
eval_dataset  = split['test']

print(f'Train: {len(train_dataset):,} examples')
print(f'Eval:  {len(eval_dataset):,} examples')


Train: 9,000 examples
Eval:  1,000 examples


## 3. Load Mistral in 4-bit

**What BitsAndBytesConfig does:**

Tells the loader to quantize the model to 4-bit as it loads.
The model never exists in full precision in memory.

```
load_in_4bit:           use 4-bit quantization
bnb_4bit_quant_type:    'nf4' = NormalFloat4, best for LLMs
bnb_4bit_compute_dtype: float16 for computation (faster than float32)
bnb_4bit_use_double_quant: quantize the quantization constants too
                           saves extra ~0.4GB
```


In [ ]:
# !pip install -U -q bitsandbytes>=0.46.1

In [ ]:
# !pip show bitsandbytes

In [ ]:
BASE_MODEL = 'mistralai/Mistral-7B-Instruct-v0.2'

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print('Loading Mistral-7B in 4-bit...')
print('This downloads ~4GB on first run')

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',        # automatically place on GPU
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Model loaded')
print(f'VRAM used: {torch.cuda.memory_reserved(0)/1024**3:.1f} GB')


Loading Mistral-7B in 4-bit...
This downloads ~4GB on first run


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Model loaded
VRAM used: 4.0 GB


## 4. Add LoRA Adapters

**The target modules explained:**

```
q_proj  = query projection  ┐
k_proj  = key projection    ├ attention layers
v_proj  = value projection  ┘  most important for style
o_proj  = output projection
```


In [ ]:
lora_config = LoraConfig(
    r=16,                    
    lora_alpha=32,           
    target_modules=[         
        'q_proj',
        'k_proj',
        'v_proj',
        'o_proj',
    ],
    lora_dropout=0.05,       
    bias='none',
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 13,631,488 || all params: 7,255,363,584 || trainable%: 0.1879


## 5. Training Arguments

**Key settings explained:**

```
num_train_epochs:           3 passes through all 9k training examples
per_device_train_batch_size: 4 examples per GPU step
gradient_accumulation_steps: 8 steps before updating weights
  effective batch size = 4 × 8 = 32
  larger effective batch = more stable training

learning_rate: 2e-4
  how fast to adjust weights per step
  too high = unstable, too low = slow learning
  2e-4 is standard for LoRA

warmup_ratio: 0.03
  first 3% of steps: learning rate ramps up slowly
  prevents instability at the start

save_steps: 500
  save checkpoint every 500 steps to Drive
  if Colab disconnects, you lose at most 500 steps

fp16: True
  use 16-bit math during training (faster, less memory)
```

In [ ]:
print(train_dataset.column_names)
print(train_dataset[0])

['text', 'ideology', 'topic', 'raw_tweet']
{'text': "<s>[INST] You are a Democratic senator speaking about climate and environment. Express your position. [/INST]\nI led and in urging the to take action on PFAS contamination including at Van Etten Lake near Oscoda. The proposed plan for remediation doesnt comprehensively address the problemI'll keep pushing for a swift, concrete clean-up plan.</s>", 'ideology': 'democrat', 'topic': 'climate and environment', 'raw_tweet': "I led and in urging the to take action on PFAS contamination including at Van Etten Lake near Oscoda. The proposed plan for remediation doesnt comprehensively address the problemI'll keep pushing for a swift, concrete clean-up plan."}


In [ ]:
OUTPUT_DIR = f'{DRIVE_PATH}/checkpoints'


## 6. Train

**What SFTTrainer does:**

SFT = Supervised Fine-Tuning.
It reads  `text` column, tokenizes it,
and trains the model to predict the next token.



In [ ]:
steps_per_epoch = len(train_dataset) // (8 * 4)
total_steps = steps_per_epoch * 3

print("Steps per epoch:", steps_per_epoch)
print("Total steps:", total_steps)

Steps per epoch: 281
Total steps: 843


In [ ]:
from trl import SFTConfig

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,

    # Dataset
    dataset_text_field="text",
    max_length=256,

    # Training
    num_train_epochs=3,

    # Batch size
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,  

    # Learning rate
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",

    # Precision / optimizer
    fp16=False,
    bf16=False,
    optim="adamw_torch",

    # Speed
    gradient_checkpointing=False,

    # Checkpoints
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,

    # Evaluation
    eval_strategy="steps",
    eval_steps=50,
    load_best_model_at_end=True,

    # Logging
    logging_steps=10,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)



print('Starting training...')
print('Watch the loss decrease — that is the model learning ideology')
print()

trainer.train()

print('Training complete')


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Starting training...
Watch the loss decrease — that is the model learning ideology



Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,1.703104,1.915994,1.618187,113386.000000,0.580433
100,1.491291,1.516221,1.522250,227405.000000,0.646333
150,1.472262,1.495507,1.528312,341534.000000,0.650305
200,1.466029,1.489525,1.506938,454200.000000,0.649786
250,1.516229,1.481945,1.536000,568105.000000,0.651618
300,1.363470,1.476968,1.484187,679341.000000,0.651950
350,1.341166,1.483735,1.426688,793168.000000,0.651453
400,1.374674,1.477057,1.422500,906165.000000,0.652337
450,1.333891,1.476093,1.408062,1019617.000000,0.654177
500,1.326275,1.472161,1.408062,1133612.000000,0.654149


In [ ]:
# Save final model
trainer.save_model(f"{OUTPUT_DIR}/final_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_model")